# Hi-C Chromatin Contact Map Analysis
**Data:** Rao et al. (2014) GM12878 lymphoblastoid cell line, 1 Mb resolution (Cooler format)  
**Methods:** Contact matrix visualization · P(s) distance-decay curves · Observed/Expected normalization · Compartment identification

---

## Overview

Hi-C is a genome-wide chromosome conformation capture technique that measures the frequency of physical contact between all pairs of genomic loci simultaneously. The result is a **contact matrix** — a symmetric, genome-wide map where each cell (i, j) represents how often genomic region i was found in physical proximity to genomic region j in the nucleus.

Hi-C data reveals higher-order chromatin organization features that are invisible to linear genomic analyses:

- **Compartments A and B** — large (~5 Mb) domains corresponding roughly to euchromatin (active, gene-rich) and heterochromatin (inactive, gene-poor) regions, visible as a checkerboard pattern in the contact matrix
- **Topologically Associating Domains (TADs)** — smaller (~1 Mb) self-interacting domains bounded by insulator elements, visible as squares along the diagonal
- **Loops** — specific long-range contacts between regulatory elements (enhancers, promoters) visible as dots off the diagonal

This notebook explores the Rao et al. (2014) Hi-C dataset from GM12878 lymphoblastoid cells — one of the landmark papers in 3D genome organization — using the **Cooler** ecosystem, which provides an efficient format and API for sparse Hi-C matrices.

**References:**
- Rao et al. (2014) *A 3D Map of the Human Genome at Kilobase Resolution Reveals Principles of Chromatin Looping.* Cell 159, 1665–1680
- Cooler documentation: https://cooler.readthedocs.io/en/latest/

## 1. Setup

We use the **Cooler** library, developed by the Mirny Lab at MIT, which provides a memory-efficient format (`.cool` files, backed by HDF5) and a Python API for working with Hi-C contact matrices. Sparse storage is essential for Hi-C data — at high resolution, the full contact matrix for the human genome would be enormous if stored densely.

In [ ]:
!pip install cooler numpy cooltools -q

In [ ]:
import cooler
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

## 2. Load the Hi-C Dataset

We download a 1 Mb resolution `.cool` file from the Rao et al. (2014) GM12878 dataset, publicly available through the open2c community repository. At 1 Mb resolution, each bin covers 1 million base pairs — lower resolution than the paper's original analysis, but suitable for seeing large-scale compartment structure and demonstrating the analysis workflow efficiently.

In [ ]:
!wget -q https://github.com/open2c/cooler-binder/raw/master/data/Rao2014-GM12878-MboI-allreps-filtered.1000kb.cool \
    -O GM12878_1Mb.cool

print("Download complete.")

In [ ]:
# Load the Cooler object — this is the primary interface for all data access
c = cooler.Cooler('GM12878_1Mb.cool')

# Inspect the dataset metadata
print("Dataset info:")
for key, val in c.info.items():
    print(f"  {key}: {val}")

In [ ]:
# The cooler object exposes three main tables:
# c.bins()   — genomic bins (chromosome, start, end)
# c.pixels() — non-zero contact counts (bin1_id, bin2_id, count)
# c.matrix() — matrix-style access to contact counts

print(f"Resolution: {c.binsize:,} bp per bin")
print(f"Total bins: {c.info['nbins']:,}")
print(f"Total non-zero pixels: {c.info['nnz']:,}")
print(f"\nChromosomes: {c.chromnames}")

## 3. Raw Contact Matrix Visualization

We start by visualizing the raw contact matrix for chromosome 6. The matrix is symmetric — entry (i, j) equals entry (j, i) — so it's conventional to display only the upper or lower triangle, though we show the full matrix here for clarity.

**What to look for:**
- The bright diagonal represents contacts between adjacent bins (nearby genomic regions interact most frequently — this is expected given polymer physics)
- Off-diagonal enrichment in a checkerboard pattern indicates A/B compartments
- The white gaps along the diagonal at the centromere reflect bins with very few mappable reads (repetitive sequences that can't be uniquely aligned)

We apply a log1p transform (`log(counts + 1)`) to compress the dynamic range, since contact frequency drops sharply with genomic distance.

In [ ]:
def plot_contact_map(chrom, cooler_obj, figsize=(10, 10), cmap='viridis', title=None):
    """
    Plot a log-normalized Hi-C contact matrix for a single chromosome.

    Parameters
    ----------
    chrom : str — chromosome name (e.g., 'chr6')
    cooler_obj : cooler.Cooler — loaded Cooler object
    figsize : tuple — figure dimensions
    cmap : str — matplotlib colormap
    title : str — optional plot title override
    """
    matrix = cooler_obj.matrix(balance=False).fetch(chrom)

    plt.figure(figsize=figsize)
    plt.imshow(np.log1p(matrix), cmap=cmap, aspect='equal')
    plt.colorbar(label='log(counts + 1)', fraction=0.046, pad=0.04)
    plt.title(title or f'Hi-C Contact Map — {chrom} (1 Mb resolution)', fontsize=13)
    plt.xlabel('Genomic bin')
    plt.ylabel('Genomic bin')
    plt.tight_layout()
    plt.show()

    return matrix  # return for downstream analysis

matrix_chr6 = plot_contact_map('chr6', c)

**Interpreting the contact map:** the checkerboard pattern visible at the scale of multiple diagonal blocks reflects A/B compartment structure. Regions of the same compartment type (A–A or B–B) interact more frequently than regions of different compartment types (A–B), producing the alternating enrichment and depletion pattern off the diagonal.

## 4. P(s) — Contact Probability as a Function of Genomic Distance

One of the most informative summary statistics of a Hi-C dataset is the **P(s) curve**: how does the average contact probability between two loci decrease as their genomic separation (s) increases?

For a polymer in equilibrium, we expect contact probability to scale with distance as a power law: P(s) ∝ s⁻ᵅ. The exponent α encodes information about the physical organization of chromatin:

- **α ≈ 1.5** — consistent with an equilibrium polymer model
- **Fractal globule model** (Lieberman-Aiden et al. 2009) predicts α ≈ 1.0 at intermediate scales

We compute P(s) by averaging along each off-diagonal of the contact matrix, where diagonal k contains all pairs of bins separated by k bins (i.e., k × bin_size base pairs).

In [ ]:
def compute_ps(matrix, bin_size):
    """
    Compute the P(s) contact probability curve from a contact matrix.

    Parameters
    ----------
    matrix : np.ndarray — square contact count matrix
    bin_size : int — size of each genomic bin in base pairs

    Returns
    -------
    distances : np.ndarray — genomic distances in bp for each diagonal
    ps : np.ndarray — mean contact probability at each distance
    """
    n = matrix.shape[0]
    diags = np.arange(n)
    ps = np.empty(n, dtype=float)

    for k in diags:
        d = np.diag(matrix, k=k)
        ps[k] = np.nanmean(d) if d.size > 0 else np.nan

    distances = diags * bin_size
    return distances, ps

In [ ]:
bin_size = c.binsize
distances_chr6, ps_chr6 = compute_ps(matrix_chr6, bin_size)

# Plot P(s) on a log-log scale — power law relationships appear as straight lines
valid = (ps_chr6 > 0) & np.isfinite(ps_chr6) & (distances_chr6 > 0)

plt.figure(figsize=(8, 5))
plt.loglog(distances_chr6[valid] / 1e6, ps_chr6[valid],
           color='steelblue', linewidth=2, label='chr6 P(s)')
plt.xlabel('Genomic distance (Mb)')
plt.ylabel('Mean contact probability P(s)')
plt.title('P(s) Distance Decay Curve — chr6\n'
          '(log-log scale; straight line = power law decay)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Fit a power law to the P(s) curve and report the exponent
# P(s) ~ s^alpha; on a log-log plot this is a straight line with slope alpha
valid_fit = valid & (distances_chr6 > bin_size)  # skip the self-contact diagonal

log_d = np.log10(distances_chr6[valid_fit])
log_p = np.log10(ps_chr6[valid_fit])

# Fit a line to the log-log data
alpha, intercept = np.polyfit(log_d, log_p, 1)
print(f"Estimated power law exponent (alpha): {alpha:.3f}")
print(f"Interpretation: P(s) decays as s^{alpha:.2f}")
print(f"  alpha ~ -1.0: consistent with fractal globule model")
print(f"  alpha ~ -1.5: consistent with equilibrium polymer model")

## 5. Observed/Expected Normalization

The raw contact matrix is dominated by the distance-decay effect — nearby bins always have high counts regardless of any specific biological interaction. To reveal compartment structure and loops, we normalize each contact count by the **expected** contact frequency at that genomic distance, computed from the P(s) curve.

The **Observed/Expected (O/E) matrix** highlights contacts that are enriched or depleted relative to the genomic-distance baseline:
- **O/E > 1** — this pair of loci contacts more than expected given their distance (enrichment)
- **O/E < 1** — this pair contacts less than expected (depletion)
- **O/E = 1** — exactly at the expected frequency

The checkerboard compartment pattern becomes much clearer in the O/E matrix than in the raw matrix.

In [ ]:
def compute_oe_matrix(matrix, bin_size):
    """
    Compute the Observed/Expected contact matrix by normalizing raw counts
    by the expected contact frequency at each genomic distance.

    Parameters
    ----------
    matrix : np.ndarray — raw contact count matrix
    bin_size : int — genomic bin size in bp

    Returns
    -------
    oe : np.ndarray — observed/expected normalized matrix
    ps_fit : np.ndarray — fitted expected contact frequency per diagonal
    """
    distances, ps = compute_ps(matrix, bin_size)
    n = matrix.shape[0]

    # Fit a power law to P(s) to use as the expected value at each distance
    valid = (ps > 0) & np.isfinite(ps) & (distances > 0)
    log_d = np.log10(distances[valid])
    log_p = np.log10(ps[valid])
    a, b = np.polyfit(log_d[1:], log_p[1:], 1)

    # Compute expected value for every diagonal
    diags = np.arange(n)
    with np.errstate(divide='ignore', invalid='ignore'):
        ps_fit = np.where(
            diags > 0,
            10 ** (a * np.log10(diags * bin_size) + b),
            np.nan
        )

    # Build the O/E matrix
    E = np.zeros_like(matrix, dtype=float)
    for k, m in zip(diags, ps_fit):
        if np.isfinite(m) and m > 0:
            np.fill_diagonal(E[k:], m)
            np.fill_diagonal(E[:, k:], m)

    oe = np.divide(matrix, E,
                   out=np.zeros_like(matrix, dtype=float),
                   where=E > 0)
    return oe, ps_fit

In [ ]:
oe_chr6, _ = compute_oe_matrix(matrix_chr6, bin_size)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Raw contact matrix
im0 = axes[0].imshow(np.log1p(matrix_chr6), cmap='viridis', aspect='equal')
axes[0].set_title('Raw Contact Matrix — chr6\n(log scale)', fontsize=12)
plt.colorbar(im0, ax=axes[0], label='log(counts + 1)', fraction=0.046)

# O/E matrix — diverging colormap (red=enriched, blue=depleted)
im1 = axes[1].imshow(oe_chr6, cmap='RdBu_r', aspect='equal',
                      vmin=0.5, vmax=1.5, origin='upper')
axes[1].set_title('Observed/Expected Matrix — chr6\n'
                   'Red = more contact than expected; Blue = less', fontsize=12)
plt.colorbar(im1, ax=axes[1], label='O/E ratio', fraction=0.046)

plt.suptitle('Chr6 Contact Map: Raw vs. Observed/Expected Normalized', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**What the O/E matrix reveals:** the checkerboard pattern in the O/E matrix reflects **A/B compartments**. In the raw matrix this pattern is hard to see because it's overwhelmed by the distance-decay effect. After O/E normalization, regions that belong to the same compartment (both A or both B) show red enrichment even at large genomic distances, while regions in different compartments show blue depletion.

## 6. Comparison — Chromosome 2

We apply the same analysis to chromosome 2 to illustrate that compartment structure varies between chromosomes. Chr2 is the longest human autosome and shows a particularly clear example of A/B compartment organization at 1 Mb resolution.

In [ ]:
matrix_chr2 = plot_contact_map('chr2', c,
    title='Hi-C Contact Map — chr2 (1 Mb resolution)')

In [ ]:
oe_chr2, _ = compute_oe_matrix(matrix_chr2, bin_size)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

im0 = axes[0].imshow(np.log1p(matrix_chr2), cmap='viridis', aspect='equal')
axes[0].set_title('Raw Contact Matrix — chr2', fontsize=12)
plt.colorbar(im0, ax=axes[0], label='log(counts + 1)', fraction=0.046)

im1 = axes[1].imshow(oe_chr2, cmap='RdBu_r', aspect='equal',
                      vmin=0.5, vmax=1.5, origin='upper')
axes[1].set_title('Observed/Expected Matrix — chr2', fontsize=12)
plt.colorbar(im1, ax=axes[1], label='O/E ratio', fraction=0.046)

plt.suptitle('Chr2 Contact Map: Raw vs. Observed/Expected Normalized', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# P(s) comparison: do chr6 and chr2 share the same distance-decay exponent?
distances_chr2, ps_chr2 = compute_ps(matrix_chr2, bin_size)

valid6 = (ps_chr6 > 0) & np.isfinite(ps_chr6) & (distances_chr6 > 0)
valid2 = (ps_chr2 > 0) & np.isfinite(ps_chr2) & (distances_chr2 > 0)

plt.figure(figsize=(8, 5))
plt.loglog(distances_chr6[valid6] / 1e6, ps_chr6[valid6],
           color='steelblue', linewidth=2, label='chr6')
plt.loglog(distances_chr2[valid2] / 1e6, ps_chr2[valid2],
           color='darkorange', linewidth=2, label='chr2')
plt.xlabel('Genomic distance (Mb)')
plt.ylabel('Mean contact probability P(s)')
plt.title('P(s) Distance Decay — chr6 vs. chr2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Summary

This notebook implemented a core Hi-C data analysis workflow:

1. **Data loading** — used the Cooler API to load a 1 Mb resolution Hi-C contact matrix from the Rao et al. (2014) GM12878 dataset
2. **Contact map visualization** — plotted log-normalized raw contact matrices for chr6 and chr2, identifying the diagonal structure, centromeric gaps, and preliminary compartment patterning
3. **P(s) analysis** — computed and plotted contact probability as a function of genomic distance on a log-log scale, fit a power law to estimate the decay exponent, and compared chr6 vs. chr2
4. **Observed/Expected normalization** — removed the distance-decay effect to reveal A/B compartment structure as a checkerboard enrichment pattern in the O/E matrix

**Key takeaways:**
- Raw Hi-C contact matrices are dominated by the distance-decay effect — normalization is required before biological structure becomes visible
- The O/E matrix reveals A/B compartments: pairs of loci in the same compartment type interact more than expected at their genomic distance, producing correlated enrichment blocks
- P(s) curves follow approximately power-law decay, with the exponent providing information about the physical polymer organization of chromatin
- The Cooler ecosystem (`.cool` format, `cooler` Python API, `cooltools`) is the standard in the field for working with Hi-C data at any resolution

**Potential extensions:**
- Compute the first eigenvector of the O/E matrix (after correlation transformation) to formally call A/B compartment assignments per bin
- Apply insulation score or TAD boundary calling to identify Topologically Associating Domains at higher resolution (e.g., 40 kb)
- Compare compartment structure between cell types or between normal and disease states
- Analyze loop calls using HICCUPS or other peak-calling methods on high-resolution data